# Reviewer Walkthrough: Branch-Resolved Cytoskeletal-Dendritic Accessibility Model

This root-level notebook is for reviewers who want to both understand the branch-resolved cytoskeletal-dendritic accessibility model and test the repository in a structured way.

It bridges the reviewer-facing guidance that is currently spread across:
- `README.md`
- `CLAIMS_TO_EXPERIMENTS.md`
- `RUN.md`
- `OUTPUTS.md`
- `notebooks/reviewer_reproduction_walkthrough.py`

## Scope boundary

This repository can reproduce the executable simulator results and the reported open-data analyses. It does **not** directly observe the proposed slow cytoskeletal accessibility field itself. That remains a biological hypothesis requiring future branch-resolved perturbation experiments.

## How to use this notebook

1. Run the setup cell first.
2. Work through the direct simulator primer to understand branch-level state variables and replay.
3. Use the guided execution sections to reproduce Level 0 and Level 1 results.
4. Use the artifact-audit and DANDI sections if you want a broader end-to-end review.

All paths and commands assume the notebook is being run from the repository root.

## Repository Map And Reviewer Path

The repository is organized around five reviewer-relevant areas:
- `src/cytodend_accessmodel/`: the executable branch-level simulator and contracts.
- `experiments/`: runnable entry points for model, ablation, comparison, and open-data analyses.
- `configs/`: DANDI dataset configuration files.
- `data/`: local raw data and derived reviewer-facing artifacts.
- `article/`: manuscript source, figures, and exported PDFs.

The recommended reviewer path mirrors `RUN.md`:
- **Level 0:** install, import/test sanity checks, seed validation, and DANDI inventories when data exist.
- **Level 1:** no-data simulator reproduction for the executable model claims.
- **Level 2:** manuscript open-data reproduction for DANDI `000336`, `000718`, and `001710`.
- **Level 3:** figure regeneration, robustness layers, and final audit passes.

## Practical note on open-data scope

The no-data simulator path is lightweight. The full DANDI-backed reproduction is optional for many reviewers and can require roughly **180-200 GB** of free disk space. This notebook keeps those sections clearly separated so you can stop after the simulator path if that is sufficient for your review.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint

# Repository naming note: the simulator package was originally named
# `cytodend_keylock` and has since been renamed to `cytodend_accessmodel`.
# The manuscript Methods section (submitted draft) still references the old
# name `src/cytodend_keylock/`; this will be corrected at the proof stage.
# All imports and experiment scripts in this repository already use the
# current name `cytodend_accessmodel`.


def detect_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return start


REPO_ROOT = detect_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Repository root: {REPO_ROOT}")
print(f"Source path added: {SRC_ROOT}")
print(f"Python executable: {sys.executable}")

Repository root: I:\AAA_PYTHON_REPOS_BACKUP\cytodendaccessmodel
Source path added: I:\AAA_PYTHON_REPOS_BACKUP\cytodendaccessmodel\src
Python executable: C:\Program Files\Python312\python.exe


## Direct Model Primer

The shortest path to understanding the model is to inspect the simulator directly rather than starting with the full experiment scripts.

The key branch-level variables are:
- `M_b`: slow structural accessibility.
- `E_b`: eligibility trace marking recent branch-specific activity.
- `P_b`: translation readiness used during consolidation.
- `A_b^f`: fast accessibility driven by cue, context, spines, and inhibition.
- `A_b^s`: slow accessibility derived from `M_b`.
- `A_b`: effective accessibility, the product of fast and slow access.

The two compact demos below are adapted from `experiments/exp001_minimal_branch_linking.py` and `experiments/exp002_context_sensitive_recall.py`.

What to look for:
- overlap branches should show the strongest replay-dependent structural change
- consolidation should increase branch-specific linking
- matching retrieval context should beat mismatched context
- post-consolidation context separation should widen relative to pre-consolidation

### Parameter note: demo set vs. canonical set

This notebook uses **two distinct parameter sets** that appear in the repository, and it is important to keep them apart:

| Parameter | Demo (this notebook) | Canonical (S2 Appendix / `exp013`) |
|---|---|---|
| `structural_gain` | 2.0 | 6.0 |
| `structural_lr` | 0.20 | 0.18 |
| `eligibility_decay` | 0.10 | 0.12 |
| `replay_gain` | 1.2 | 0.80 |
| `sleep_gain` | 0.8 | 0.0 |
| Consolidation passes | 3 | 9 (3 nights × 3) |
| Cue inputs (b1, b2) | b1=0.9, b2=0.05 | b1=0.8, b2=0.0 |

The **demo set** (used in the two interactive cells below) is the same as `experiments/exp001_minimal_branch_linking.py`. It produces the representative numbers cited in manuscript **Table 2**: `ΔM_b1 = +0.21758` and a `43.1 %` rise in the linking metric.

The **canonical set** (used in `experiments/exp013_paper_summary.py` and documented in S2 Appendix Table 5) is the parameter set for the detailed `M_b` trajectory tables and the paper-level aggregate. Running `exp013` with 9 passes per condition gives `ΔM_b1 ≈ +0.299` and a `58.6 %` linking rise — different numbers, same qualitative direction.

The two sets exist because `exp001` was the original minimal demo and `exp013` was added later to create a more conservative canonical reference for S2. Both are correct; the distinction matters when comparing output numbers with the manuscript.


In [2]:
from cytodend_accessmodel import (
    ConsolidationWindow,
    CytodendAccessModelSimulator,
    DynamicsParameters,
    EngramTrace,
    TraceAllocation,
)


def print_rows(rows: list[dict[str, object]], columns: list[str]) -> None:
    widths = {}
    for col in columns:
        content_width = max((len(str(row.get(col, ""))) for row in rows), default=0)
        widths[col] = max(len(col), content_width)
    header = "  ".join(f"{col:<{widths[col]}}" for col in columns)
    divider = "  ".join("-" * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print("  ".join(f"{str(row.get(col, '')):<{widths[col]}}" for col in columns))


def load_json(path: Path):
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def summarize_json_shape(value) -> str:
    if value is None:
        return "missing"
    if isinstance(value, list):
        return f"list[{len(value)}]"
    if isinstance(value, dict):
        keys = ", ".join(list(value)[:6])
        suffix = "..." if len(value) > 6 else ""
        return f"dict[{len(value)}]: {keys}{suffix}"
    return type(value).__name__


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


BRANCH_IDS = ["b0", "b1", "b2", "b3"]

MU1_ALLOCATION = TraceAllocation(
    trace_id="mu1",
    branch_weights={"b0": 0.9, "b1": 0.85, "b2": 0.05, "b3": 0.05},
)
MU2_ALLOCATION = TraceAllocation(
    trace_id="mu2",
    branch_weights={"b0": 0.05, "b1": 0.85, "b2": 0.9, "b3": 0.05},
)
MU1_CUE = {"b0": 1.0, "b1": 0.9, "b2": 0.05, "b3": 0.0}
MU2_CUE = {"b0": 0.05, "b1": 0.9, "b2": 1.0, "b3": 0.0}

ALPHA_ALLOCATION = TraceAllocation(
    trace_id="mu_alpha",
    branch_weights={"b0": 0.9, "b1": 0.8, "b2": 0.05, "b3": 0.05},
)
BETA_ALLOCATION = TraceAllocation(
    trace_id="mu_beta",
    branch_weights={"b0": 0.05, "b1": 0.05, "b2": 0.8, "b3": 0.9},
)
ALPHA_CONTEXT_BIAS = {"b0": 0.7, "b1": 0.6, "b2": 0.0, "b3": 0.0}
BETA_CONTEXT_BIAS = {"b0": 0.0, "b1": 0.0, "b2": 0.6, "b3": 0.7}
ALPHA_CUE = {"b0": 1.0, "b1": 0.9, "b2": 0.05, "b3": 0.0}
BETA_CUE = {"b0": 0.0, "b1": 0.05, "b2": 0.9, "b3": 1.0}
PARTIAL_CUE = {"b0": 0.4, "b1": 0.4, "b2": 0.4, "b3": 0.4}


def branch_rows(sim: CytodendAccessModelSimulator) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for branch_id in BRANCH_IDS:
        branch = sim.branches[branch_id]
        rows.append(
            {
                "branch": branch_id,
                "M_b": f"{branch.structural.accessibility:.4f}",
                "E_b": f"{branch.eligibility.value:.4f}",
                "P_b": f"{branch.translation_readiness.value:.4f}",
                "A_f": f"{branch.fast_access:.4f}",
                "A_s": f"{branch.slow_access:.4f}",
                "A": f"{branch.effective_access:.4f}",
                "x_b": f"{branch.activation:.4f}",
            }
        )
    return rows


def recall_supports(sim: CytodendAccessModelSimulator) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for support in sim.compute_recall_supports():
        rows.append(
            {
                "trace": support.trace_id,
                "support": f"{support.support:.4f}",
                "expressed": f"{support.expressed_strength:.4f}",
                "active_branches": ", ".join(support.active_branches) or "-",
            }
        )
    return rows


def compute_linking(sim: CytodendAccessModelSimulator) -> float:
    return sum(
        MU1_ALLOCATION.branch_weights.get(branch_id, 0.0)
        * MU2_ALLOCATION.branch_weights.get(branch_id, 0.0)
        * sim.branches[branch_id].structural.accessibility
        for branch_id in BRANCH_IDS
    )


def build_branch_linking_sim() -> CytodendAccessModelSimulator:
    params = DynamicsParameters(
        fast_gain=2.0,
        structural_gain=2.0,
        eligibility_decay=0.1,
        translation_decay=0.05,
        structural_lr=0.20,
        structural_decay=0.005,
        structural_max=1.0,
        replay_gain=1.2,
        sleep_gain=0.8,
        readout_gain=5.0,
        readout_threshold=0.3,
    )
    sim = CytodendAccessModelSimulator.from_branch_ids(
        BRANCH_IDS, spines_per_branch=3, parameters=params
    )
    sim.add_trace(EngramTrace(trace_id="mu1", allocation=MU1_ALLOCATION, label="trace-mu1"))
    sim.add_trace(EngramTrace(trace_id="mu2", allocation=MU2_ALLOCATION, label="trace-mu2"))
    return sim


def build_context_sim() -> CytodendAccessModelSimulator:
    params = DynamicsParameters(
        fast_gain=2.0,
        context_gain=1.0,
        structural_gain=2.0,
        eligibility_decay=0.1,
        translation_decay=0.05,
        structural_lr=0.20,
        structural_decay=0.005,
        structural_max=1.0,
        replay_gain=1.2,
        sleep_gain=0.8,
        readout_gain=4.0,
        readout_threshold=0.4,
        context_mismatch_penalty=0.35,
    )
    sim = CytodendAccessModelSimulator.from_branch_ids(
        BRANCH_IDS, spines_per_branch=3, parameters=params
    )
    sim.add_trace(
        EngramTrace(
            trace_id="mu_alpha",
            allocation=ALPHA_ALLOCATION,
            label="alpha-trace",
            context="alpha",
        )
    )
    sim.add_trace(
        EngramTrace(
            trace_id="mu_beta",
            allocation=BETA_ALLOCATION,
            label="beta-trace",
            context="beta",
        )
    )
    return sim

In [3]:
sim = build_branch_linking_sim()

for _ in range(2):
    sim.apply_cue(MU1_CUE)
for _ in range(2):
    sim.apply_cue(MU2_CUE)

pre_linking = compute_linking(sim)
pre_rows = branch_rows(sim)

print("Pre-consolidation branch state")
print_rows(pre_rows, ["branch", "M_b", "E_b", "P_b", "A_f", "A_s", "A", "x_b"])

print("\nPre-consolidation recall with cue mu1")
sim.apply_cue(MU1_CUE)
print_rows(recall_supports(sim), ["trace", "support", "expressed", "active_branches"])

print("\nPre-consolidation recall with cue mu2")
sim.apply_cue(MU2_CUE)
print_rows(recall_supports(sim), ["trace", "support", "expressed", "active_branches"])

for pass_idx in range(3):
    report = sim.run_consolidation(
        ConsolidationWindow(
            window_id=f"sleep-pass-{pass_idx}",
            modulatory_drive=1.0,
            sleep_drive=1.0,
            replay_trace_ids=("mu1", "mu2"),
        )
    )
    print(
        f"\nConsolidation pass {pass_idx}: "
        f"branches_updated={report.branches_updated}, "
        f"mean_shift={report.mean_structural_shift:.5f}, "
        f"mean_P_b={report.mean_translation_readiness:.5f}"
    )

post_linking = compute_linking(sim)
post_rows = branch_rows(sim)

print("\nPost-consolidation branch state")
print_rows(post_rows, ["branch", "M_b", "E_b", "P_b", "A_f", "A_s", "A", "x_b"])

summary_rows = []
for pre_row, post_row in zip(pre_rows, post_rows):
    branch_id = pre_row["branch"]
    pre_m = float(pre_row["M_b"])
    post_m = float(post_row["M_b"])
    role = {
        "b1": "overlap branch",
        "b0": "single-trace branch",
        "b2": "single-trace branch",
        "b3": "unrelated branch",
    }[branch_id]
    summary_rows.append(
        {
            "branch": branch_id,
            "pre_M_b": f"{pre_m:.4f}",
            "post_M_b": f"{post_m:.4f}",
            "delta": f"{post_m - pre_m:+.4f}",
            "role": role,
        }
    )

print("\nReplay-dependent structural change")
print_rows(summary_rows, ["branch", "pre_M_b", "post_M_b", "delta", "role"])
print(f"\nLinking pre:  {pre_linking:.5f}")
print(f"Linking post: {post_linking:.5f}")
print(f"Linking delta:{post_linking - pre_linking:+.5f}")

Pre-consolidation branch state
branch  M_b     E_b     P_b     A_f     A_s     A       x_b   
------  ------  ------  ------  ------  ------  ------  ------
b0      0.5000  0.8548  0.0000  0.6457  0.7311  0.4720  0.0236
b1      0.5000  1.0000  0.0000  0.9089  0.7311  0.6644  0.5980
b2      0.5000  1.0000  0.0000  0.9241  0.7311  0.6756  0.6756
b3      0.5000  0.0000  0.0000  0.6225  0.7311  0.4551  0.0000

Pre-consolidation recall with cue mu1
trace  support  expressed  active_branches
-----  -------  ---------  ---------------
mu1    1.1175   0.9835     b0, b1         
mu2    0.5633   0.7886     b0, b1         

Pre-consolidation recall with cue mu2
trace  support  expressed  active_branches
-----  -------  ---------  ---------------
mu2    1.1175   0.9835     b1, b2         
mu1    0.5633   0.7886     b1, b2         

Consolidation pass 0: branches_updated=4, mean_shift=0.06819, mean_P_b=0.75000

Consolidation pass 1: branches_updated=4, mean_shift=0.05085, mean_P_b=0.79875

Consolid

In [4]:
def recall_under_contexts(sim: CytodendAccessModelSimulator) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    scenarios = [
        ("alpha", ALPHA_CONTEXT_BIAS),
        ("beta", BETA_CONTEXT_BIAS),
        ("none", {}),
    ]
    for context_name, bias in scenarios:
        active_context = None if context_name == "none" else context_name
        sim.apply_cue(PARTIAL_CUE, context=active_context, context_bias=bias)
        support_map = {item["trace"]: item for item in recall_supports(sim)}
        alpha_support = float(support_map["mu_alpha"]["support"])
        beta_support = float(support_map["mu_beta"]["support"])
        winner = "mu_alpha" if alpha_support > beta_support else "mu_beta"
        if alpha_support == beta_support:
            winner = "tie"
        rows.append(
            {
                "context": context_name,
                "R(mu_alpha)": f"{alpha_support:.4f}",
                "R(mu_beta)": f"{beta_support:.4f}",
                "winner": winner,
            }
        )
    return rows


sim = build_context_sim()

for _ in range(3):
    sim.apply_cue(ALPHA_CUE, context="alpha", context_bias=ALPHA_CONTEXT_BIAS)
for _ in range(3):
    sim.apply_cue(BETA_CUE, context="beta", context_bias=BETA_CONTEXT_BIAS)

print("Structural state before replay-driven consolidation")
print_rows(branch_rows(sim), ["branch", "M_b", "E_b", "P_b", "A_f", "A_s", "A", "x_b"])

pre_context_rows = recall_under_contexts(sim)
print("\nRecall under different contexts before consolidation")
print_rows(pre_context_rows, ["context", "R(mu_alpha)", "R(mu_beta)", "winner"])

for pass_idx in range(3):
    sim.run_consolidation(
        ConsolidationWindow(
            window_id=f"context-sleep-{pass_idx}",
            modulatory_drive=1.0,
            sleep_drive=1.0,
            replay_trace_ids=("mu_alpha", "mu_beta"),
        )
    )

print("\nStructural state after replay-driven consolidation")
print_rows(branch_rows(sim), ["branch", "M_b", "E_b", "P_b", "A_f", "A_s", "A", "x_b"])

post_context_rows = recall_under_contexts(sim)
print("\nRecall under different contexts after consolidation")
print_rows(post_context_rows, ["context", "R(mu_alpha)", "R(mu_beta)", "winner"])

comparison_rows = []
for pre_row, post_row in zip(pre_context_rows, post_context_rows):
    comparison_rows.append(
        {
            "context": pre_row["context"],
            "pre_alpha": pre_row["R(mu_alpha)"],
            "post_alpha": post_row["R(mu_alpha)"],
            "pre_beta": pre_row["R(mu_beta)"],
            "post_beta": post_row["R(mu_beta)"],
        }
    )

print("\nPre/post context comparison")
print_rows(comparison_rows, ["context", "pre_alpha", "post_alpha", "pre_beta", "post_beta"])
print("\nExpected reading: matching context should win, and the gap should usually widen after consolidation.")

Structural state before replay-driven consolidation
branch  M_b     E_b     P_b     A_f     A_s     A       x_b   
------  ------  ------  ------  ------  ------  ------  ------
b0      0.5000  0.7290  0.0000  0.6225  0.7311  0.4551  0.0000
b1      0.5000  0.7930  0.0000  0.6457  0.7311  0.4720  0.0236
b2      0.5000  1.0000  0.0000  0.9478  0.7311  0.6929  0.6236
b3      0.5000  1.0000  0.0000  0.9608  0.7311  0.7024  0.7024

Recall under different contexts before consolidation
context  R(mu_alpha)  R(mu_beta)  winner  
-------  -----------  ----------  --------
alpha    0.4583       0.2706      mu_alpha
beta     0.2706       0.4583      mu_beta 
none     0.2689       0.4136      mu_beta 

Structural state after replay-driven consolidation
branch  M_b     E_b     P_b     A_f     A_s     A       x_b   
------  ------  ------  ------  ------  ------  ------  ------
b0      0.7142  0.7290  1.0000  0.7858  0.8066  0.6339  0.2298
b1      0.7074  0.7290  1.0000  0.7858  0.8045  0.6322  0.22

## Guided Execution: Minimal Recommended Path

If you want the shortest reviewer path that still covers the branch-resolved executable model, start here.

Recommended first-pass order:
- install the package in editable mode with `.[dev,viz]`
- run `pytest`
- run `experiments/exp_seed_validation.py`
- run `experiments/exp001_minimal_branch_linking.py`
- run `experiments/exp002_context_sensitive_recall.py`
- run `experiments/exp013_paper_summary.py`

This path gives you:
- environment sanity
- a canonical RNG-state check
- a direct branch-linking example
- a direct context-sensitive recall example
- the canonical reviewer JSON summary at `data/reviewer/013_canonical_values.json`

The cell below prints the commands in runnable order. Uncomment the loop only if you want the notebook to execute them from the repository root.

In [5]:
import subprocess


def run_command(command: str) -> int:
    print(f"\n$ {command}")
    completed = subprocess.run(command, cwd=REPO_ROOT, shell=True, check=False)
    print(f"exit code: {completed.returncode}")
    return completed.returncode


minimal_reviewer_commands = [
    'python -m pip install --upgrade pip',
    'python -m pip install -e ".[dev,viz]"',
    'pytest',
    'python experiments/exp_seed_validation.py',
    'python experiments/exp001_minimal_branch_linking.py',
    'python experiments/exp002_context_sensitive_recall.py',
    'python experiments/exp013_paper_summary.py',
]

print("Minimal reviewer path:")
for command in minimal_reviewer_commands:
    print(f"- {command}")

# Uncomment to run the entire minimal path from this notebook.
# for command in minimal_reviewer_commands:
#     run_command(command)

Minimal reviewer path:
- python -m pip install --upgrade pip
- python -m pip install -e ".[dev,viz]"
- pytest
- python experiments/exp_seed_validation.py
- python experiments/exp001_minimal_branch_linking.py
- python experiments/exp002_context_sensitive_recall.py
- python experiments/exp013_paper_summary.py


## Full Level 1 Simulator Reproduction

`RUN.md` defines Level 1 as the no-data simulator reproduction layer. These scripts cover the core model properties, robustness, pathology, rescue, topology, richer readout, ablations, comparisons, and executable figures.

Use this section if you want to audit the simulator claims more broadly than the minimal reviewer path.

Primary outputs at this layer:
- terminal verdicts and summary tables from most `experiments/exp*.py` scripts
- `data/reviewer/013_canonical_values.json` from `experiments/exp013_paper_summary.py`
- executable SVG figures from `experiments/gen_figures_executable.py`

The cell below lists the full Level 1 command order used by the repo guidance.

In [6]:
level_1_commands = [
    'python experiments/exp001_minimal_branch_linking.py',
    'python experiments/exp002_context_sensitive_recall.py',
    'python experiments/exp003_timing_replay_linking.py',
    'python experiments/exp004_robustness.py',
    'python experiments/exp005_pathology.py',
    'python experiments/exp006_asymmetric_consolidation.py',
    'python experiments/exp007_branch_heterogeneity.py',
    'python experiments/exp008_local_competition.py',
    'python experiments/exp009_rescue_linking.py',
    'python experiments/exp010_multitrace_overlap.py',
    'python experiments/exp011_branch_topology.py',
    'python experiments/exp012_retrieval_readout.py',
    'python experiments/exp014_structural_gate_ablation.py',
    'python experiments/exp015_comparator_baselines.py',
    'python experiments/exp016_task_family.py',
    'python experiments/exp013_paper_summary.py',
    'python experiments/gen_figures_executable.py',
]

print("Full Level 1 command list:")
for command in level_1_commands:
    print(f"- {command}")

# Uncomment to execute the full Level 1 simulator layer.
# for command in level_1_commands:
#     run_command(command)

Full Level 1 command list:
- python experiments/exp001_minimal_branch_linking.py
- python experiments/exp002_context_sensitive_recall.py
- python experiments/exp003_timing_replay_linking.py
- python experiments/exp004_robustness.py
- python experiments/exp005_pathology.py
- python experiments/exp006_asymmetric_consolidation.py
- python experiments/exp007_branch_heterogeneity.py
- python experiments/exp008_local_competition.py
- python experiments/exp009_rescue_linking.py
- python experiments/exp010_multitrace_overlap.py
- python experiments/exp011_branch_topology.py
- python experiments/exp012_retrieval_readout.py
- python experiments/exp014_structural_gate_ablation.py
- python experiments/exp015_comparator_baselines.py
- python experiments/exp016_task_family.py
- python experiments/exp013_paper_summary.py
- python experiments/gen_figures_executable.py


## Claim-To-Output Audit

Use this section to connect manuscript-facing claims to concrete scripts, outputs, and interpretation boundaries.

High-level reviewer checklist:
- `H1` structural perturbation proxy: `exp005`, `exp009`, and `exp014` test vulnerability, rescue, and ablation behavior in the simulator.
- `H2` memory linking and replay timing: `exp001`, `exp003`, `exp010`, and `exp011` cover overlap, replay dependence, multi-trace overlap, and topology effects.
- `H3` contextual retrieval: `exp002`, `exp012`, and `exp015` test context-sensitive recall and comparator behavior.
- canonical aggregation: `exp013` writes `data/reviewer/013_canonical_values.json`.
- open-data bridge for `000718`: the main reviewer-facing endpoint is `data/dandi/triage/000718/h1_pri_enrichment.json` plus the robustness and specificity outputs.
- open-data bridge for `000336`: the main reviewer-facing endpoint is `data/dandi/triage/000336/full_bundle_coupling.json`.
- open-data bridge for `001710`: the main reviewer-facing endpoints live under `data/dandi/triage/001710/robustness/` and `data/dandi/triage/001710/replication_bundle/`.

Interpretation boundaries to keep in view:
- simulator outputs reproduce executable model signatures, not direct molecular validation
- `000718` is a constrained enrichment bridge, not sequence-level replay proof
- `000336` supports structured access-constraint interpretations, not direct slow-field measurement
- `001710` is bounded by channel sensitivity and indirect labeling assumptions

In [7]:
def status(path: Path) -> str:
    return "present" if path.exists() else "missing"


def print_artifact_status(title: str, paths: list[Path]) -> None:
    print(f"\n{title}")
    print("-" * len(title))
    for path in paths:
        print(f"{status(path):>7}  {rel(path)}")


def dataset_presence(dataset_id: str) -> dict[str, object]:
    raw_dir = REPO_ROOT / "data" / "dandi" / "raw" / dataset_id
    triage_dir = REPO_ROOT / "data" / "dandi" / "triage" / dataset_id
    raw_count = len(list(raw_dir.rglob("*.nwb"))) if raw_dir.exists() else 0
    triage_count = len([path for path in triage_dir.rglob("*") if path.is_file()]) if triage_dir.exists() else 0
    return {
        "dataset": dataset_id,
        "raw_dir": status(raw_dir),
        "raw_nwb_files": raw_count,
        "triage_dir": status(triage_dir),
        "triage_files": triage_count,
    }


artifacts = {
    "Simulator canonical summary": [
        REPO_ROOT / "data" / "reviewer" / "013_canonical_values.json",
    ],
    "DANDI 000718 offline enrichment": [
        REPO_ROOT / "data" / "dandi" / "triage" / "000718" / "h1_pri_enrichment.json",
        REPO_ROOT / "data" / "dandi" / "triage" / "000718" / "h1_pri_enrichment.md",
        REPO_ROOT / "data" / "dandi" / "triage" / "000718" / "h1_robustness.json",
        REPO_ROOT / "data" / "dandi" / "triage" / "000718" / "h1_specificity.json",
    ],
    "DANDI 000336 full-bundle coupling": [
        REPO_ROOT / "data" / "dandi" / "triage" / "000336" / "full_bundle_coupling.json",
        REPO_ROOT / "data" / "dandi" / "triage" / "000336" / "full_bundle_coupling.md",
    ],
    "DANDI 001710 robustness and nulls": [
        REPO_ROOT / "data" / "dandi" / "triage" / "001710" / "robustness" / "group_null_tests.json",
        REPO_ROOT / "data" / "dandi" / "triage" / "001710" / "robustness" / "claim_boundary.md",
        REPO_ROOT / "data" / "dandi" / "triage" / "001710" / "robustness" / "day_lag_similarity.md",
    ],
}

for title, paths in artifacts.items():
    print_artifact_status(title, paths)

canonical_json = REPO_ROOT / "data" / "reviewer" / "013_canonical_values.json"
canonical = load_json(canonical_json)
print(f"\nCanonical reviewer JSON: {rel(canonical_json)} -> {summarize_json_shape(canonical)}")
if isinstance(canonical, dict):
    pprint({key: summarize_json_shape(value) for key, value in canonical.items()})

presence_rows = [dataset_presence(dataset_id) for dataset_id in ("000336", "000718", "001710")]
print("\nDataset presence summary")
print_rows(presence_rows, ["dataset", "raw_dir", "raw_nwb_files", "triage_dir", "triage_files"])

print("\nUse missing rows as a guide: missing raw data blocks Level 2 computation, while missing triage outputs usually mean the matching experiment series has not been run yet.")


Simulator canonical summary
---------------------------
present  data\reviewer\013_canonical_values.json

DANDI 000718 offline enrichment
-------------------------------
present  data\dandi\triage\000718\h1_pri_enrichment.json
present  data\dandi\triage\000718\h1_pri_enrichment.md
present  data\dandi\triage\000718\h1_robustness.json
present  data\dandi\triage\000718\h1_specificity.json

DANDI 000336 full-bundle coupling
---------------------------------
present  data\dandi\triage\000336\full_bundle_coupling.json
present  data\dandi\triage\000336\full_bundle_coupling.md

DANDI 001710 robustness and nulls
---------------------------------
present  data\dandi\triage\001710\robustness\group_null_tests.json
present  data\dandi\triage\001710\robustness\claim_boundary.md
present  data\dandi\triage\001710\robustness\day_lag_similarity.md

Canonical reviewer JSON: data\reviewer\013_canonical_values.json -> dict[6]: canonical_params, mb_trajectory, core_metrics, pathology, rescue, claims
{'cano

## Optional DANDI Open-Data Workflow

This section is optional and high-cost. Use it only if you want to audit the manuscript's open-data bridge rather than the no-data simulator alone.

Current-manuscript datasets:
- `000336`: structured cross-plane coupling analysis
- `000718`: offline enrichment and robustness analysis
- `001710`: subject-level robustness, nulls, and replication bundle outputs

Open-data reviewer flow:
1. confirm you want to spend the disk space and runtime budget
2. download only the dataset(s) needed for the claim you want to audit
3. run the matching numbered Level 2 scripts in order
4. inspect the derived outputs under `data/dandi/triage/`
5. optionally regenerate manuscript-facing open-data figures with `python -m dandi_analysis.visualisation.cli`

The next cell prints the Level 2 command groups and the most important output locations for each dataset.

In [8]:
dandi_download_commands = [
    'dandi download --output-dir data/dandi/raw "DANDI:000336/sub-644972"',
    'dandi download --output-dir data/dandi/raw "DANDI:000336/sub-656228"',
    'dandi download --output-dir data/dandi/raw "DANDI:000718/sub-Ca-EEG2-1"',
    'dandi download --output-dir data/dandi/raw "DANDI:000718/sub-Ca-EEG3-4"',
    'dandi download --output-dir data/dandi/raw "DANDI:001710"',
]

level_2_groups = {
    '000336': [
        'python experiments/dandi_000336_01_inventory.py',
        'python experiments/dandi_000336_02_header_probe.py',
        'python experiments/dandi_000336_03_crossplane_coupling.py',
        'python experiments/dandi_000336_04_sub656228_replication.py',
        'python experiments/dandi_000336_05_ses1245548523.py',
        'python experiments/dandi_000336_06_full_bundle.py',
    ],
    '000718': [
        'python experiments/dandi_000718_01_inventory.py',
        'python experiments/dandi_000718_02_header_probe.py',
        'python experiments/dandi_000718_03_offline_epoch_candidates.py',
        'python experiments/dandi_000718_04_activity_matrix_smoke_test.py',
        'python experiments/dandi_000718_05_pairwise_coreactivation_baseline.py',
        'python experiments/dandi_000718_06_ensemble_reactivation.py',
        'python experiments/dandi_000718_08_h1_neutral_offline.py',
        'python experiments/dandi_000718_10_h1_event_registration.py',
        'python experiments/dandi_000718_11_h1_robustness.py',
        'python experiments/dandi_000718_12_h1_specificity.py',
        'python experiments/dandi_000718_13_h1_pri.py',
        'python experiments/dandi_000718_14_h1_pri_enrichment.py',
    ],
    '001710': [
        'python experiments/dandi_001710_01_inventory.py',
        'python experiments/dandi_001710_02_header_probe.py',
        'python experiments/dandi_001710_03_trial_reconstruction.py',
        'python experiments/dandi_001710_04_activity_matrix_smoke_test.py',
        'python experiments/dandi_001710_05_within_day_place_tuning.py',
        'python experiments/dandi_001710_06_cross_day_remapping_baseline.py',
        'python experiments/dandi_001710_07_replication_bundle_full_pass.py',
        'python experiments/dandi_001710_08_robustness_and_nulls.py',
    ],
}

primary_outputs = {
    '000336': [
        'data/dandi/triage/000336/full_bundle_coupling.json',
        'data/dandi/triage/000336/full_bundle_coupling.md',
        'article/A branch-resolved cytoskeletal-dendritic accessibility model of associative memory/figures/figure_8_open_data_000336_coupling_by_condition.png',
        'article/A branch-resolved cytoskeletal-dendritic accessibility model of associative memory/figures/figure_9_open_data_000336_replication.png',
    ],
    '000718': [
        'data/dandi/triage/000718/h1_pri_enrichment.json',
        'data/dandi/triage/000718/h1_robustness.json',
        'data/dandi/triage/000718/h1_specificity.json',
        'article/A branch-resolved cytoskeletal-dendritic accessibility model of associative memory/figures/figure_6_open_data_000718_enrichment.png',
        'article/A branch-resolved cytoskeletal-dendritic accessibility model of associative memory/figures/figure_7_open_data_000718_threshold_sweep.png',
    ],
    '001710': [
        'data/dandi/triage/001710/replication_bundle/',
        'data/dandi/triage/001710/robustness/group_null_tests.json',
        'data/dandi/triage/001710/robustness/claim_boundary.md',
        'data/dandi/triage/001710/robustness/day_lag_similarity.md',
    ],
}

print('Representative DANDI download commands:')
for command in dandi_download_commands:
    print(f'- {command}')

for dataset_id in ('000336', '000718', '001710'):
    print(f'\nDataset {dataset_id} Level 2 commands:')
    for command in level_2_groups[dataset_id]:
        print(f'  - {command}')
    print('Primary reviewer-facing outputs:')
    for path in primary_outputs[dataset_id]:
        print(f'  - {path}')

print('\nFigure regeneration command:')
print('- python -m dandi_analysis.visualisation.cli')

Representative DANDI download commands:
- dandi download --output-dir data/dandi/raw "DANDI:000336/sub-644972"
- dandi download --output-dir data/dandi/raw "DANDI:000336/sub-656228"
- dandi download --output-dir data/dandi/raw "DANDI:000718/sub-Ca-EEG2-1"
- dandi download --output-dir data/dandi/raw "DANDI:000718/sub-Ca-EEG3-4"
- dandi download --output-dir data/dandi/raw "DANDI:001710"

Dataset 000336 Level 2 commands:
  - python experiments/dandi_000336_01_inventory.py
  - python experiments/dandi_000336_02_header_probe.py
  - python experiments/dandi_000336_03_crossplane_coupling.py
  - python experiments/dandi_000336_04_sub656228_replication.py
  - python experiments/dandi_000336_05_ses1245548523.py
  - python experiments/dandi_000336_06_full_bundle.py
Primary reviewer-facing outputs:
  - data/dandi/triage/000336/full_bundle_coupling.json
  - data/dandi/triage/000336/full_bundle_coupling.md
  - article/A branch-resolved cytoskeletal-dendritic accessibility model of associative memo

## Wrap-Up And Validation Checklist

Use this notebook as a running audit log.

What you can verify without any DANDI download:
- the package imports from `src/`
- the direct branch-linking and context-sensitive simulator behavior
- the Level 0 sanity path
- the Level 1 no-data simulator claims
- the existence and shape of `data/reviewer/013_canonical_values.json`

What still depends on external data:
- the `000336`, `000718`, and `001710` open-data results
- their derived triage outputs under `data/dandi/triage/`
- the manuscript-facing open-data figures regenerated from those outputs

A practical end-of-review checklist:
- early explanatory cells run without requiring NWB files
- missing DANDI data shows up as missing directories or artifacts rather than crashing the notebook
- direct simulator demos show branch-specific replay effects and context-sensitive retrieval
- root-relative commands match the documented workflow in `README.md` and `RUN.md`
- optional DANDI sections remain clearly separate from the lightweight simulator review path